# Explore the single lens microlensing model

In [ ]:
import MulensModel as mm
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def compute_flux(t_range, t_0, u_0, t_E, log_rho, F_total, f_s):
    rho = 10**log_rho
    model = mm.Model({'t_0': t_0, 'u_0': u_0, 't_E': t_E, 'rho': rho})
    model.set_magnification_methods(
        [t_range[0], 'finite_source_uniform_Gould94', t_range[-1]]
    )
    mag = model.get_magnification(t_range)
    return F_total * (1 + f_s * (mag - 1))

def plot_both(t0_a, u0_a, tE_a, logrho_a, Ft_a, fs_a,
              t0_b, u0_b, tE_b, logrho_b, Ft_b, fs_b):
    t_lo = min(t0_a - 5 * tE_a, t0_b - 5 * tE_b)
    t_hi = max(t0_a + 5 * tE_a, t0_b + 5 * tE_b)
    t = np.linspace(t_lo, t_hi, 7200)

    flux_a = compute_flux(t, t0_a, u0_a, tE_a, logrho_a, Ft_a, fs_a)
    flux_b = compute_flux(t, t0_b, u0_b, tE_b, logrho_b, Ft_b, fs_b)

    residual = flux_b - flux_a

    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=(10, 6), height_ratios=[3, 1],
        sharex=True, gridspec_kw={'hspace': 0.05}
    )

    ax_top.plot(t, flux_a, color='#a859e4', lw=2, label='Model A')
    ax_top.plot(t, flux_b, color='#3dbf8a', lw=2, ls='--', label='Model B')
    ax_top.set_ylabel('Flux')
    ax_top.legend(loc='upper right')

    ax_bot.axhline(0, color='grey', lw=0.8)
    ax_bot.plot(t, residual, color='#e05555', lw=1.2)
    ax_bot.set_xlabel('Time')
    ax_bot.set_ylabel('B $-$ A')

    plt.tight_layout()
    plt.show()

style = {'description_width': '60px'}
layout = widgets.Layout(width='420px')

def make_sliders(tag, color, defaults):
    header = widgets.HTML(
        f"<b style='color:{color}; font-size:14px;'>Model {tag}</b>"
    )
    s_t0  = widgets.FloatSlider(value=defaults['t_0'], min=-50, max=50, step=0.1,
                                description='t_0', style=style, layout=layout)
    s_u0  = widgets.FloatSlider(value=defaults['u_0'], min=0.01, max=2.0, step=0.01,
                                description='u_0', readout_format='.4f',
                                style=style, layout=layout)
    s_tE  = widgets.FloatSlider(value=defaults['t_E'], min=1, max=100, step=0.5,
                                description='t_E', style=style, layout=layout)
    s_logrho = widgets.FloatSlider(value=defaults['log_rho'], min=-5, max=0, step=0.01,
                                   description='log ρ', readout_format='.2f',
                                   style=style, layout=layout)
    s_Ft  = widgets.FloatSlider(value=defaults['F_tot'], min=0.1, max=100, step=0.1,
                                description='F_tot', style=style, layout=layout)
    s_fs  = widgets.FloatSlider(value=defaults['f_s'], min=0.0, max=1.0, step=0.01,
                                description='f_s', style=style, layout=layout)
    box = widgets.VBox([header, s_t0, s_u0, s_tE, s_logrho, s_Ft, s_fs])
    return box, {'t_0': s_t0, 'u_0': s_u0, 't_E': s_tE,
                 'log_rho': s_logrho, 'F_tot': s_Ft, 'f_s': s_fs}

defaults_a = dict(t_0=0, u_0=0.1, t_E=10, log_rho=-3, F_tot=10.0, f_s=1.0)
defaults_b = dict(t_0=0, u_0=0.1, t_E=10, log_rho=-3, F_tot=10.0, f_s=1.0)

box_a, sliders_a = make_sliders('A', '#a859e4', defaults_a)
box_b, sliders_b = make_sliders('B', '#3dbf8a', defaults_b)

out = widgets.interactive_output(plot_both, {
    't0_a': sliders_a['t_0'], 'u0_a': sliders_a['u_0'],
    'tE_a': sliders_a['t_E'], 'logrho_a': sliders_a['log_rho'],
    'Ft_a': sliders_a['F_tot'], 'fs_a': sliders_a['f_s'],
    't0_b': sliders_b['t_0'], 'u0_b': sliders_b['u_0'],
    'tE_b': sliders_b['t_E'], 'logrho_b': sliders_b['log_rho'],
    'Ft_b': sliders_b['F_tot'], 'fs_b': sliders_b['f_s'],
})

controls = widgets.HBox([box_a, box_b])
display(widgets.VBox([controls, out]))